In [1]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import sys

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [2]:
module_path = "/home/ubuntu/Shree_FYP/train/stage2/training"

In [3]:
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
processor = AutoProcessor.from_pretrained("/home/ubuntu/Shree_FYP/data/stage1_unsloth")

The tokenizer you are loading from '/home/ubuntu/Shree_FYP/data/stage1_unsloth' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [5]:
teacher = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path = "/home/ubuntu/Shree_FYP/data/stage1_unsloth",
    dtype = torch.bfloat16,
    attn_implementation = "flash_attention_2",
    trust_remote_code = True
    )

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [6]:
hasattr(teacher.model, "visual")

True

In [7]:
teacher.model.visual

Qwen3_5VisionModel(
  (patch_embed): Qwen3_5VisionPatchEmbed(
    (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
  )
  (pos_embed): Embedding(2304, 1024)
  (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
  (blocks): ModuleList(
    (0-23): 24 x Qwen3_5VisionBlock(
      (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
      (attn): Qwen3_5VisionAttention(
        (qkv): Linear(in_features=1024, out_features=3072, bias=True)
        (proj): Linear(in_features=1024, out_features=1024, bias=True)
      )
      (mlp): Qwen3_5VisionMLP(
        (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
        (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
        (act_fn): GELUTanh()
      )
    )
  )
  (merger): Qwen3_5VisionPatchMerger(
    (norm): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
    (linear_fc1): Linear(in_features=4096, out_f

In [8]:
from peft import LoraConfig, TaskType, get_peft_model, get_peft_model_state_dict

In [9]:
lora_cfg = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r = 64,
    lora_alpha = 128,
    lora_dropout = 0.05,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "out_proj", "in_proj_qkv", "in_proj_z", "in_proj_b", "in_proj_a", "gate_proj", "up_proj", "down_proj"],
    bias = "none"
)

In [10]:
vlm = get_peft_model(teacher, lora_cfg)

In [11]:
inner_model = vlm.model.model

In [12]:
vlm.config.image_token_id

248056

In [13]:
_ref_model = vlm

In [18]:
_ref_model.model.lm_head

Linear(in_features=2560, out_features=248320, bias=False)

In [20]:
vlm.model.model.language_model.embed_tokens

Embedding(248320, 2560)

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 